# v13 2D-U-Net 学習 ― PF画像から TVT を予測(5-fold)

`build_images_v13.py` が出力した `artifacts_v13/images/{well}.npz`(順/逆密度 + GRコストの 3ch画像)を入力に、
2D-U-Net で **各MD列の TVTオフセット**を予測する。

- 出力: U-Net が (H,T) ヒートマップ → 各MD列で H方向(TVT)softmax → **soft-argmax** で連続オフセット(ft)。
  予測 TVT = `center + オフセット`。`target_rel`(=真TVT−center)基準で評価 → これは TVT RMSE と一致。
- 学習: T方向を **ランダムクロップ**(長い坑井のVRAM対策)。損失 = マスク付きクロスエントロピー + soft-argmax MSE。
- 評価: **GroupKFold(5) by well**、坑井ごと全長推論 → OOF 全行プールの **overall RMSE**。
- OOF は `artifacts_v13/oof_unet.npz`(well→pred_rel)に保存。後で v10/v12 とブレンド可能。

In [1]:
import time, copy, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

In [2]:
# ============================== CONFIG ==============================
PROJECT  = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
IMG_DIR  = PROJECT / "v13" / "artifacts_v13" / "images"     # build_images_v13.py の出力
OUT_DIR  = PROJECT / "v13" / "artifacts_v13"
N_SPLITS = 5
SEED     = 0
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

CROP     = 1024      # 学習時の T方向クロップ長(VRAM対策)
HIDDEN   = 32        # U-Net 基本チャンネル
EPOCHS   = 30
BATCH    = 4
LR       = 1e-3
WD       = 1e-5
PATIENCE = 6
CE_W     = 1.0       # クロスエントロピー重み
MSE_W    = 0.05      # soft-argmax 回帰の補助重み(ft^2 スケール)
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

device: cuda NVIDIA GeForce RTX 5070


In [3]:
# 画像メタの確認 & 坑井リスト
import pandas as pd
wells = sorted(p.stem for p in IMG_DIR.glob("*.npz"))
assert len(wells) > 0, f"画像が無い: {IMG_DIR} (先に build_images_v13.py を実行)"
d0 = np.load(IMG_DIR / f"{wells[0]}.npz")
C, H = d0["img"].shape[0], d0["img"].shape[1]
BINW = float(d0["binw"]); BAND = float(d0["band"])
HAS_T = "target_rel" in d0.files
print(f"wells={len(wells)}  C={C} H={H}  band=±{BAND}ft binw={BINW}  has_target={HAS_T}")
assert HAS_T, "target_rel が無い(train画像で実行すること)"

wells=773  C=7 H=129  band=±32.0ft binw=0.5  has_target=True


In [4]:
def load_well(wid):
    d = np.load(IMG_DIR / f"{wid}.npz")
    img = d["img"].astype(np.float32)                 # (C,H,T)
    tr  = d["target_rel"].astype(np.float32)          # (T,) 真TVT−center(ft)
    msk = d["mask"].astype(np.float32) if "mask" in d.files else (np.abs(tr) <= BAND).astype(np.float32)
    tbin = np.clip(np.round(tr / BINW).astype(np.int64) + H // 2, 0, H - 1)   # (T,) 正解ビン
    return img, tr, msk, tbin

class CropDS(Dataset):
    """坑井ごとに T方向ランダムクロップ(短い坑井はパディング+マスク)。"""
    def __init__(self, well_list):
        self.wells = list(well_list)
    def __len__(self): return len(self.wells)
    def __getitem__(self, i):
        img, tr, msk, tbin = load_well(self.wells[i])
        T = img.shape[2]
        if T >= CROP:
            s = np.random.randint(0, T - CROP + 1)
            sl = slice(s, s + CROP)
            return (torch.from_numpy(img[:, :, sl]), torch.from_numpy(tbin[sl]),
                    torch.from_numpy(msk[sl]))
        # pad to CROP
        ip = np.zeros((C, H, CROP), np.float32); ip[:, :, :T] = img
        bp = np.zeros(CROP, np.int64); bp[:T] = tbin
        mp = np.zeros(CROP, np.float32); mp[:T] = msk
        return torch.from_numpy(ip), torch.from_numpy(bp), torch.from_numpy(mp)

In [5]:
# 2D U-Net(3レベル, デコーダは interpolate でスキップ形状に合わせる→奇数サイズOK)
class DoubleConv(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.GELU(),
            nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.GELU())
    def forward(self, x): return self.net(x)

class UNet2D(nn.Module):
    def __init__(self, C, h):
        super().__init__()
        self.e1 = DoubleConv(C, h); self.e2 = DoubleConv(h, h*2); self.e3 = DoubleConv(h*2, h*4)
        self.b  = DoubleConv(h*4, h*4)
        self.d3 = DoubleConv(h*4 + h*4, h*2); self.d2 = DoubleConv(h*2 + h*2, h); self.d1 = DoubleConv(h + h, h)
        self.head = nn.Conv2d(h, 1, 1)
        self.pool = nn.MaxPool2d(2, ceil_mode=True)
    def _up(self, x, ref): return F.interpolate(x, size=ref.shape[-2:], mode="nearest")
    def forward(self, x):                      # (B,C,H,T)
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b  = self.b(self.pool(e3))
        d3 = self.d3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.d2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.d1(torch.cat([self._up(d2, e1), e1], 1))
        return self.head(d1).squeeze(1)        # (B,H,T) ロジット(H=TVTビン)

_BINS = None
def soft_argmax_offset(logits):
    """logits:(B,H,T) -> 連続オフセット(ft) (B,T)。期待値ビン→ft。"""
    global _BINS
    if _BINS is None or _BINS.shape[1] != logits.shape[1]:
        _BINS = torch.arange(logits.shape[1], device=logits.device, dtype=logits.dtype).view(1, -1, 1)
    prob = torch.softmax(logits, dim=1)
    exp_bin = (prob * _BINS).sum(1)            # (B,T)
    return (exp_bin - logits.shape[1] // 2) * BINW

In [6]:
def predict_well_rel(model, wid, device):
    """坑井全長推論 → 予測オフセット(ft) (T,)。"""
    img, _, _, _ = load_well(wid)
    x = torch.from_numpy(img).unsqueeze(0).to(device)     # (1,C,H,T)
    model.eval()
    with torch.no_grad():
        off = soft_argmax_offset(model(x)).squeeze(0).float().cpu().numpy()
    return off

def overall_rmse_wells(model, well_list, device):
    se = 0.0; n = 0
    for w in well_list:
        _, tr, _, _ = load_well(w)
        pr = predict_well_rel(model, w, device)
        se += float(np.sum((pr - tr) ** 2)); n += len(tr)
    return math.sqrt(se / max(n, 1))

def run_fold(tr_wells, va_wells, device, verbose=True):
    loader = DataLoader(CropDS(tr_wells), batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)
    model = UNet2D(C, HIDDEN).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    best = float("inf"); best_state = None; patience = 0
    for ep in range(EPOCHS):
        model.train()
        for img, tbin, msk in loader:
            img, tbin, msk = img.to(device), tbin.to(device), msk.to(device)
            logits = model(img)                              # (B,H,T)
            ce = F.cross_entropy(logits, tbin, reduction="none")          # (B,T)
            ce = (ce * msk).sum() / msk.sum().clamp_min(1.0)
            off = soft_argmax_offset(logits)                 # (B,T) ft
            tgt_ft = (tbin.float() - H // 2) * BINW
            mse = (((off - tgt_ft) ** 2) * msk).sum() / msk.sum().clamp_min(1.0)
            loss = CE_W * ce + MSE_W * mse
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        vr = overall_rmse_wells(model, va_wells, device)
        if vr < best - 1e-4:
            best = vr; best_state = copy.deepcopy(model.state_dict()); patience = 0
        else:
            patience += 1
        if verbose: print(f"    epoch {ep:2d}: val_overall_rmse={vr:.4f} (best={best:.4f})")
        if patience >= PATIENCE:
            if verbose: print(f"    early stop @ {ep}")
            break
    model.load_state_dict(best_state)
    return {w: predict_well_rel(model, w, device) for w in va_wells}, best

In [7]:
# 5-fold GroupKFold by well
wells_arr = np.array(wells)
gkf = GroupKFold(n_splits=N_SPLITS)
oof = {}; fold_rmse = []
t0 = time.perf_counter()
for k, (tr, va) in enumerate(gkf.split(wells_arr, groups=wells_arr)):
    print(f"\n===== fold {k} (train={len(tr)}, val={len(va)}) =====")
    preds, best = run_fold(wells_arr[tr], wells_arr[va], DEVICE)
    oof.update(preds); fold_rmse.append(best)
    print(f"  fold {k} best overall RMSE = {best:.4f}")
print(f"\nfolds done {(time.perf_counter()-t0)/60:.1f} min")


===== fold 0 (train=618, val=155) =====
    epoch  0: val_overall_rmse=8.8679 (best=8.8679)
    epoch  1: val_overall_rmse=9.0732 (best=8.8679)
    epoch  2: val_overall_rmse=8.9037 (best=8.8679)
    epoch  3: val_overall_rmse=8.8830 (best=8.8679)
    epoch  4: val_overall_rmse=8.8654 (best=8.8654)
    epoch  5: val_overall_rmse=8.7792 (best=8.7792)
    epoch  6: val_overall_rmse=8.7607 (best=8.7607)
    epoch  7: val_overall_rmse=9.0859 (best=8.7607)
    epoch  8: val_overall_rmse=8.6972 (best=8.6972)
    epoch  9: val_overall_rmse=8.9024 (best=8.6972)
    epoch 10: val_overall_rmse=8.7630 (best=8.6972)
    epoch 11: val_overall_rmse=8.9606 (best=8.6972)
    epoch 12: val_overall_rmse=8.5988 (best=8.5988)
    epoch 13: val_overall_rmse=8.9257 (best=8.5988)
    epoch 14: val_overall_rmse=8.6733 (best=8.5988)
    epoch 15: val_overall_rmse=8.7986 (best=8.5988)
    epoch 16: val_overall_rmse=8.8328 (best=8.5988)
    epoch 17: val_overall_rmse=8.7716 (best=8.5988)
    epoch 18: val_overa

In [8]:
# overall RMSE(全坑井OOFをプール)
se = 0.0; n = 0
for w in wells:
    _, tr, _, _ = load_well(w)
    pr = oof[w]
    se += float(np.sum((pr - tr) ** 2)); n += len(tr)
overall = math.sqrt(se / n)
# 参考: 坑井平均RMSE
pw = np.mean([math.sqrt(np.mean((oof[w] - load_well(w)[1]) ** 2)) for w in wells])

np.savez_compressed(OUT_DIR / "oof_unet.npz", **{w: oof[w].astype(np.float32) for w in wells})
print("fold RMSE:", [round(r,4) for r in fold_rmse])
print(f"mean fold RMSE = {np.mean(fold_rmse):.4f}")
print(f"OVERALL RMSE   = {overall:.4f}   (target_rel基準 = TVT RMSE)")
print(f"per-well mean  = {pw:.4f}")
print(f"oof saved -> {OUT_DIR/'oof_unet.npz'}  (well->pred_rel)")
print("参考: v10(LGBM,2)=7.5537 / v12(TCN)=7.8710 / ブレンド=7.3924")

fold RMSE: [8.5988, 8.2637, 7.3321, 6.7412, 10.3162]
mean fold RMSE = 8.2504
OVERALL RMSE   = 8.3373   (target_rel基準 = TVT RMSE)
per-well mean  = 6.2773
oof saved -> C:\Users\kosaka256\Documents\rogii_claude\v13\artifacts_v13\oof_unet.npz  (well->pred_rel)
参考: v10(LGBM,2)=7.5537 / v12(TCN)=7.8710 / ブレンド=7.3924
